# bag0 v5: calibration fit B (combined loss) at grid 144 on the two squeezes
Adam on normalised parameters (log scale for E and yield), both squeezes per update (one f64 adjoint each, ~16 GB: use A100/L4), loss = shape descriptors (relative to first frame, late-weighted) + original loss (frame-referenced depth). Viscosity and adhesion fixed. Set `START` to the sweep's second-best set (from `/workspace/runs/bag0/v5/sweep/top_sets_g144.json` on the GPU host; the host session pastes it). Outputs: `MyDrive/bag0_colab/fit_B/history.jsonl` and `state.json` (resumable with `--resume`).

In [ ]:
BRANCH = 'bounded-shear-adhesion-v3'
GRID = 144
START = {'youngs_modulus': 20000.0, 'von_mises_yield_stress_pa': 1000.0, 'viscosity': 1.0}  # <- replace with set 2 from top_sets_g144.json
UPDATES = 40
LR = 0.05

In [ ]:
!nvidia-smi -L
!pip -q install taichi==1.7.4 open3d==0.20.0 scipy
import taichi, open3d; print(taichi.__version__, open3d.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/bag0_colab/fit_B /content/scripts
!ls -la /content/drive/MyDrive/bag0_dynamics_v5_gpu_package.zip

In [ ]:
%cd /content
!rm -rf TaichiDough && git clone -q --branch $BRANCH https://github.com/AntonioSkara132/TaichiDough.git
%cd /content/TaichiDough
!git log --oneline -1
import zipfile
z = zipfile.ZipFile('/content/drive/MyDrive/bag0_dynamics_v5_gpu_package.zip')
z.extractall('/content/TaichiDough', [n for n in z.namelist() if n.startswith('experiments/')])
print('extracted')

In [ ]:
%%writefile /content/scripts/depth_reference.patch
diff --git a/experiments/differentiable_mpm/loss.py b/experiments/differentiable_mpm/loss.py
index 44f55fe..412cc17 100644
--- a/experiments/differentiable_mpm/loss.py
+++ b/experiments/differentiable_mpm/loss.py
@@ -44,11 +44,17 @@ class LossConfig:
     min_predicted_pixels: int = 1
     max_observed_points: int = 1024
     missing_depth_residual: float = 4.0
+    # "initial": depth change since the first frame, on pixels covered by the initial render
+    # (fails once the dough leaves its initial footprint). "frame": current observed depth vs
+    # current rendered depth on every observed pixel; shape-sensitive on transported dough.
+    depth_reference: str = "initial"
     version: str = LOSS_VERSION
 
     def __post_init__(self):
         if self.version not in SUPPORTED_LOSS_VERSIONS:
             raise ValueError(f"Unsupported training loss version: {self.version}")
+        if self.depth_reference not in ("initial", "frame"):
+            raise ValueError("depth_reference must be 'initial' or 'frame'")
         positive = ("visibility_temperature_m", "opacity_gain", "visible_cutoff_temperatures",
                     "depth_scale_m", "distance_scale_m", "huber_delta", "missing_depth_residual")
         for name in positive:
@@ -149,9 +155,13 @@ class ObservationLoss:
             if self.depth_mask[row, column] == 1:
                 term = self._huber(ti.cast(self.config.missing_depth_residual, self.fp))
                 if self.renderer.valid[row, column] == 1:
-                    observed_change = self.observed_depth[row, column] - self.observed_initial_depth[row, column]
-                    simulation_change = self.renderer.depth[row, column] - self.initial_simulation_depth[row, column]
-                    term = self._huber((simulation_change - observed_change) / self.config.depth_scale_m)
+                    if ti.static(self.config.depth_reference == "frame"):
+                        residual = self.renderer.depth[row, column] - self.observed_depth[row, column]
+                        term = self._huber(residual / self.config.depth_scale_m)
+                    else:
+                        observed_change = self.observed_depth[row, column] - self.observed_initial_depth[row, column]
+                        simulation_change = self.renderer.depth[row, column] - self.initial_simulation_depth[row, column]
+                        term = self._huber((simulation_change - observed_change) / self.config.depth_scale_m)
                 if ti.static(self.config.version == "partial-visible-splats-v2"):
                     coverage = self.renderer.coverage[row, column]
                     missing = self._huber(ti.cast(self.config.missing_depth_residual, self.fp))
@@ -191,10 +201,13 @@ class ObservationLoss:
         observed_count = int(masks[0].sum())
         if observed_count < self.config.min_observed_pixels:
             raise InvalidStateError(f"Observed support {observed_count} is below {self.config.min_observed_pixels}")
-        depth_mask = masks[0] & masks[1] & self.initial_render.valid
+        if self.config.depth_reference == "frame":
+            depth_mask = masks[0].copy()
+        else:
+            depth_mask = masks[0] & masks[1] & self.initial_render.valid
         depth_count = int(depth_mask.sum())
         if self.config.depth_weight > 0 and depth_count < self.config.min_depth_pixels:
-            raise InvalidStateError(f"Fixed initial-referenced depth support {depth_count} is below {self.config.min_depth_pixels}")
+            raise InvalidStateError(f"{'Frame-referenced' if self.config.depth_reference == 'frame' else 'Fixed initial-referenced'} depth support {depth_count} is below {self.config.min_depth_pixels}")
         points = observation.observed_points
         if points is None:
             points = self.camera.optical_points(arrays[0], masks[0])


In [ ]:
import pathlib
p = pathlib.Path('experiments/differentiable_mpm/sdf_cache.py'); s = p.read_text()
old = '    return hashlib.sha256(_json(description).encode()).hexdigest(), description'
new = '\n'.join(['    def _pin(v):', '        if isinstance(v, dict): return {k: ("2.2.6" if k == "numpy_version" else "0.17.0" if k == "open3d_version" else _pin(x)) for k, x in v.items()}', '        if isinstance(v, list): return [_pin(x) for x in v]', '        return v', '    description = _pin(description)', old])
assert old in s; p.write_text(s.replace(old, new, 1))
p = pathlib.Path('experiments/differentiable_mpm/data.py'); s = p.read_text()
old = '            settings = qualification_settings(config.tool_sdf_qualification, grid_spacing)'
new = '            grid_spacing = 1.0 / 48  # tolerance pinned to grid 48 (shipped 256^3 fields)\n' + old
assert old in s; p.write_text(s.replace(old, new, 1))
# frame-referenced depth option (ported from the other session's loss.py edit)
import subprocess; patch_text = open('/content/scripts/depth_reference.patch').read() if pathlib.Path('/content/scripts/depth_reference.patch').exists() else ''
print('patched; depth patch present:', bool(patch_text))

In [ ]:
!cd /content/TaichiDough && git apply /content/scripts/depth_reference.patch && grep -c depth_reference experiments/differentiable_mpm/loss.py

In [ ]:
import json
D = 'experiments/differentiable_mpm/data/bag0_dynamics_v5/'
d = json.load(open(D + 'dataset_bag0_all_von_mises_1000pa_coulomb_v5.json')); f = GRID // 48; eps = []
for e in d['episodes']:
    c = json.load(open(D + e['config'])); c['simulation']['grid'] = GRID
    c['simulation']['grid_box_lower'] = [v * f for v in c['simulation']['grid_box_lower']]; c['simulation']['grid_box_cells'] = [v * f for v in c['simulation']['grid_box_cells']]
    new = e['config'].replace('_coulomb_v5.json', '_coulomb_v5_g%d.json' % GRID); json.dump(c, open(D + new, 'w'), indent=2); eps.append(dict(e, config=new))
dd = dict(d); dd['name'] += '-grid%dbox' % GRID; dd['episodes'] = eps; json.dump(dd, open(D + 'dataset_v5_g%d.json' % GRID, 'w'), indent=2); print('ok')

In [ ]:
%%writefile /content/scripts/shape_loss.py
"""Shape-descriptor loss wrapper for TaichiDough rollouts (no change to the experiment tree).

Descriptors per frame: soft width along x, soft width along z (soft max - soft min, temperature tau), soft height
(soft max of y). Recorded descriptors come from the recorded clouds in the scene frame. The loss is the mean over
scored frames of sum_d ((d_sim - d_rec) / scale)^2 (optionally on changes relative to the first scored frame, which
cancels a constant reconstruction offset), weighted per frame, plus `orig_weight` times the original observation
loss. Returns LossRecord(value, dL/dpositions, components, diagnostics) like ObservationLoss."""
import numpy as np

def _band_mean(c, lo_frac, hi_frac, sign=1.0):
    """Mean of the values ranked between lo_frac and hi_frac from the top (sign=+1) or bottom (sign=-1); gradient 1/k on
    the band members (piecewise linear, robust to the extreme (1-hi_frac) fraction of outliers)."""
    n = c.shape[0]; order = np.argsort(sign * c); lo, hi = int(lo_frac * n), int(hi_frac * n)  # ascending order of sign*c: top band = ranks [lo_frac, hi_frac)
    idx = order[lo:hi]; k = len(idx); m = float(c[idx].mean()); g = np.zeros_like(c); g[idx] = 1.0 / k
    return m, g

def descriptors(x, tau=None, band=(0.90, 0.975)):
    """x: (N,3) scene-frame positions. d = [width_x, width_z, height] from band means on each side; dd/dx (3,N,3)."""
    d = np.zeros(3); g = np.zeros((3, x.shape[0], 3))
    for j, axis in enumerate((0, 2)):
        mx, gmx = _band_mean(x[:, axis], *band, +1.0); mn, gmn = _band_mean(x[:, axis], *band, -1.0)
        d[j] = mx - mn; g[j, :, axis] = gmx - gmn
    mh, gh = _band_mean(x[:, 1], *band, +1.0); d[2] = mh; g[2, :, 1] = gh
    return d, g

class ShapeDescriptorLoss:
    def __init__(self, base_loss, recorded, *, tau=0.004, scale=0.01, shape_weight=1.0, orig_weight=0.0, relative=False, frame_weights=None, initial_positions=None):
        self.base, self.rec, self.tau, self.scale = base_loss, recorded, tau, scale  # rec: {frame_index: np.array(3,)}
        self.ws, self.wo, self.relative, self.fw = shape_weight, orig_weight, relative, frame_weights or {}
        self.d0_sim = descriptors(np.asarray(initial_positions, dtype=np.float64), tau)[0] if (relative and initial_positions is not None) else None
        self.d0_rec = recorded.get(0) if relative else None

    def value_and_grad_positions(self, positions, observation, compute_grad=True):
        x = np.asarray(positions, dtype=np.float64); fi = int(observation.frame_index)
        d, g = descriptors(x, self.tau); target = self.rec[fi]
        if self.relative: d = d - self.d0_sim; target = target - self.d0_rec
        r = (d - target) / self.scale; w = self.fw.get(fi, 1.0)
        shape_value = w * float((r * r).sum()); grad = (w * 2.0 / self.scale) * np.einsum("k,kni->ni", r, g) if compute_grad else None
        components = {"shape": shape_value, "width_x_mm": float(d[0] * 1000), "width_z_mm": float(d[1] * 1000), "height_mm": float(d[2] * 1000), "rec_width_x_mm": float(target[0] * 1000), "rec_width_z_mm": float(target[1] * 1000), "rec_height_mm": float(target[2] * 1000)}
        value = self.ws * shape_value; diagnostics = {"loss_version": "shape-descriptors-v1", "frame_index": fi, "step": int(observation.step), "frame_weight": w}
        gradient = self.ws * grad if compute_grad else None
        if self.wo > 0:
            rec = self.base.value_and_grad_positions(positions, observation, compute_grad=compute_grad)
            value += self.wo * rec.value; components.update({"orig_" + k: v for k, v in rec.components.items()}); diagnostics.update({"orig_" + k: v for k, v in rec.diagnostics.items()})
            if compute_grad: gradient = gradient + self.wo * np.asarray(rec.gradient, dtype=np.float64)
        from experiments.differentiable_mpm.loss import LossRecord
        return LossRecord(float(value), None if gradient is None else gradient.astype(np.asarray(positions).dtype), components, diagnostics)

if __name__ == "__main__":  # finite-difference check of the descriptor gradient
    rng = np.random.default_rng(0); x = rng.normal(size=(2000, 3)) * 0.03; tau = None
    d, g = descriptors(x, tau); eps = 1e-7; err = 0
    for _ in range(40):
        i, a = rng.integers(2000), rng.integers(3); xp = x.copy(); xp[i, a] += eps; dp = descriptors(xp, tau)[0]
        num = (dp - d) / eps; err = max(err, np.abs(num - g[:, i, a]).max())
    print("descriptors (mm):", np.round(d * 1000, 2), "| max |FD - analytic| gradient error:", err)


In [ ]:
%%writefile /content/scripts/fit_shape.py
"""Calibration fit with the combined loss (shape descriptors relative to first frame, late-weighted, + original loss
with frame-referenced depth) using Adam on normalised parameters (log scale for E and yield), both parts per update.
usage: fit_shape.py ROOT DATASET OUT_DIR --parts P1 P2 --start '{"youngs_modulus":..,...}' [--updates 40] [--lr 0.05]
       [--orig-weight 1.0] [--fix viscosity,tool_adhesion] [--validate P3 P4] [--resume]"""
import json, os, sys, time, math, argparse, numpy as np, torch
ap = argparse.ArgumentParser(); ap.add_argument("root"); ap.add_argument("dataset"); ap.add_argument("out")
ap.add_argument("--parts", nargs="+", required=True); ap.add_argument("--start", required=True); ap.add_argument("--updates", type=int, default=40); ap.add_argument("--lr", type=float, default=0.05)
ap.add_argument("--orig-weight", type=float, default=1.0); ap.add_argument("--fix", default="viscosity,tool_adhesion"); ap.add_argument("--validate", nargs="*", default=[]); ap.add_argument("--validate-every", type=int, default=10); ap.add_argument("--resume", action="store_true"); ap.add_argument("--precision", default="f64")
a = ap.parse_args(); os.chdir(a.root); sys.path.insert(0, a.root); sys.path.insert(0, "/content/scripts"); os.makedirs(a.out, exist_ok=True)
from experiments.differentiable_mpm.calibrate import make_rollout
from experiments.differentiable_mpm.data import prepare_experiment
from experiments.differentiable_mpm.dataset_config import load_dataset
from experiments.differentiable_mpm.reference_adapter import reference_policy
from experiments.differentiable_mpm.runtime import init_runtime
from experiments.differentiable_mpm.loss_options import parse_loss_config
import dataclasses
from shape_loss import ShapeDescriptorLoss, descriptors
D = os.path.dirname(a.dataset) + "/"; T = np.asarray(json.load(open(D + "scene_calibration_v2.json"))["scene_from_source"])
LOGP = ("youngs_modulus", "von_mises_yield_stress_pa")
def ext(p): return (np.percentile(p[:, 0], 95) - np.percentile(p[:, 0], 5), np.percentile(p[:, 2], 95) - np.percentile(p[:, 2], 5), np.percentile(p[:, 1], 99))

with reference_policy("frozen"):
    ds = load_dataset(a.dataset, backend="cuda", precision=a.precision, p2g_mode="atomic"); eps = {e.id: e for e in ds.episodes}
    init_runtime("cuda", a.precision, cpu_threads=1, seed=eps[a.parts[0]].config.seed)
    bounds = dict(ds.parameter_bounds) if hasattr(ds, "parameter_bounds") else {}
    bounds.setdefault("von_mises_yield_stress_pa", [100.0, 20000.0]); bounds.setdefault("youngs_modulus", [2000.0, 60000.0])
    fixed = set(a.fix.split(",")) if a.fix else set()
    fit_names = [n for n in ("youngs_modulus", "poisson_ratio", "viscosity", "tool_friction_coefficient", "tool_adhesion", "floor_friction_coefficient", "von_mises_yield_stress_pa") if n not in fixed]
    start = dict(ds.shared_initial); start.update(json.loads(a.start))
    def to_u(p): return np.array([(math.log(p[n]) - math.log(bounds[n][0])) / (math.log(bounds[n][1]) - math.log(bounds[n][0])) if n in LOGP else (p[n] - bounds[n][0]) / (bounds[n][1] - bounds[n][0]) for n in fit_names])
    def to_p(u):
        p = dict(start)
        for n, v in zip(fit_names, np.clip(u, 0.0, 1.0)): p[n] = math.exp(math.log(bounds[n][0]) + v * (math.log(bounds[n][1]) - math.log(bounds[n][0]))) if n in LOGP else bounds[n][0] + v * (bounds[n][1] - bounds[n][0])
        return p
    def dL_du(p, g): return np.array([g.get(n, 0.0) * (p[n] * (math.log(bounds[n][1]) - math.log(bounds[n][0])) if n in LOGP else (bounds[n][1] - bounds[n][0])) for n in fit_names])
    # rollouts with the combined loss
    rollouts, recs, probes = {}, {}, {}
    for pid in a.parts + list(a.validate):
        ep = eps[pid]; cfg = ep.config
        cfg = dataclasses.replace(cfg, loss={**cfg.loss, "depth_reference": "frame"}) if isinstance(cfg.loss, dict) else cfg
        prepared = prepare_experiment(cfg, split=ep.membership, build_sdf=True, scored_window=ep.scored_window); _, ro = make_rollout(prepared, progress=lambda *x, **k: None)
        clouds = torch.load(D + f"recordings/{pid}/pointclouds_interpolated.pt", map_location="cpu", weights_only=True)[0]
        rec = {}; pts = {}
        for i in range(len(clouds)):
            q = np.asarray(clouds[i])[:, :3].astype(np.float64); q = q[np.isfinite(q).all(1)] @ T[:3, :3].T + T[:3, 3]; rec[i] = descriptors(q, 0.004)[0]; pts[i] = q
        frames = sorted(o.frame_index for o in ro.observations); n = len(frames); fw = {f: 2.0 * (k + 1) / n for k, f in enumerate(frames)}
        ro.loss = ShapeDescriptorLoss(ro.loss, rec, tau=0.004, scale=0.01, shape_weight=1.0, orig_weight=a.orig_weight, relative=True, frame_weights=fw, initial_positions=np.asarray(ro.initial_state.x, dtype=np.float64))
        rollouts[pid] = (ep, ro); recs[pid] = pts; print(f"prepared {pid}: {prepared.total_steps} steps, {n} scored frames", flush=True)
    state_path = os.path.join(a.out, "state.json"); hist_path = os.path.join(a.out, "history.jsonl")
    if a.resume and os.path.exists(state_path):
        st = json.load(open(state_path)); u = np.array(st["u"]); m = np.array(st["m"]); v = np.array(st["v"]); it0 = st["iteration"]; best = st["best"]
    else:
        u = to_u(start); m = np.zeros_like(u); v = np.zeros_like(u); it0 = 0; best = {"value": float("inf")}
    b1, b2, epsl = 0.9, 0.999, 1e-8
    def validate(p):
        out = {}
        for pid in a.validate:
            ep, ro = rollouts[pid]; sim_ext = {}
            def cb(step, state): sim_ext[step] = ext(np.asarray(state.x, dtype=np.float64))
            try:
                ev = ro.value_and_gradient(ep.parameters_for({k: p[k] for k in ds.shared_initial}) | {"von_mises_yield_stress_pa": p["von_mises_yield_stress_pa"]}, compute_grad=False, frame_callback=cb, frame_steps=sorted(ro.by_step))
                steps = sorted(sim_ext); zw = [sim_ext[s][1] * 1000 for s in steps]; hs = [sim_ext[s][2] * 1000 for s in steps]
                rec_pts = recs[pid]; rz = [ext(rec_pts[i])[1] * 1000 for i in sorted(rec_pts)]; rh = [ext(rec_pts[i])[2] * 1000 for i in sorted(rec_pts)]
                out[pid] = {"loss": float(ev.value), "sim_width_min_end": (min(zw), zw[-1]), "rec_width_min_end": (min(rz), rz[-1]), "sim_height_max": max(hs), "rec_height_max": max(rh)}
            except Exception as exc: out[pid] = {"error": str(exc)[:200]}
        return out
    for it in range(it0 + 1, a.updates + 1):
        p = to_p(u); t0 = time.perf_counter(); total = 0.0; grad = {n: 0.0 for n in fit_names}; per_part = {}
        for pid in a.parts:
            ep, ro = rollouts[pid]; params = ep.parameters_for({k: p[k] for k in ds.shared_initial}); params["von_mises_yield_stress_pa"] = p["von_mises_yield_stress_pa"]
            try: ev = ro.value_and_gradient(params, compute_grad=True)
            except Exception as exc:
                per_part[pid] = {"error": str(exc)[:200]}; print(f"update {it}: {pid} FAILED {exc}", flush=True); continue
            total += float(ev.value) / len(a.parts); per_part[pid] = {"value": float(ev.value), "shape": float(np.mean([f["components"].get("shape", 0.0) for f in ev.frames]))}
            for n in fit_names: grad[n] += float(ev.gradient.get(n, 0.0)) / len(a.parts)
        g = dL_du(p, grad); m = b1 * m + (1 - b1) * g; v = b2 * v + (1 - b2) * g * g
        mh = m / (1 - b1 ** it); vh = v / (1 - b2 ** it); u = np.clip(u - a.lr * mh / (np.sqrt(vh) + epsl), 0.0, 1.0)
        row = {"iteration": it, "value": total, "parameters": {n: p[n] for n in fit_names}, "per_part": per_part, "grad_u": g.tolist(), "seconds": round(time.perf_counter() - t0, 1)}
        if total < best["value"]: best = {"value": total, "iteration": it, "parameters": {n: p[n] for n in fit_names}}
        if a.validate and (it % a.validate_every == 0 or it == a.updates): row["validation"] = validate(p)
        with open(hist_path, "a") as f: f.write(json.dumps(row) + "\n")
        json.dump({"u": u.tolist(), "m": m.tolist(), "v": v.tolist(), "iteration": it, "best": best, "next_parameters": to_p(u)}, open(state_path, "w"), indent=1)
        print(f"update {it:3d}: loss {total:.4f} | " + " ".join(f"{n[:6]}={p[n]:.4g}" for n in fit_names) + f" | {row['seconds']:.0f} s" + (f" | val {row['validation']}" if "validation" in row else ""), flush=True)
    print("DONE best:", best, flush=True)


In [ ]:
import os, json, subprocess
os.environ['TAICHIDOUGH_SDF_CACHE'] = '/content/TaichiDough/experiments/differentiable_mpm/data/bag0_dynamics_v5_sdf_cache'
ROOT = '/content/TaichiDough'; DS = 'experiments/differentiable_mpm/data/bag0_dynamics_v5/dataset_v5_g%d.json' % GRID; OUT = '/content/drive/MyDrive/bag0_colab/fit_B'
args = ['python', '-u', '/content/scripts/fit_shape.py', ROOT, DS, OUT, '--parts', 'bag0_p06_E1.6_squeeze', 'bag0_p07_E2.1_squeeze', '--start', json.dumps(START), '--updates', str(UPDATES), '--lr', str(LR), '--validate', 'bag0_p02_E1.2_push', 'bag0_p10_E4.2_knead', '--validate-every', '10']
if os.path.exists(OUT + '/state.json'): args.append('--resume')
with open(OUT + '/fit_B.log', 'a') as f:
    proc = subprocess.Popen(args, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        f.write(line); f.flush()
        if line.startswith('update') or 'DONE' in line or 'FAILED' in line or 'prepared' in line: print(line.rstrip())
print('exit', proc.wait())